# Response Streaming with the Claude API

In standard API calls, the client waits until the model finishes generating the entire response before receiving any data. For complex prompts or lengthy generations, this can take **10 to 30 seconds**—creating a poor user experience if users are left staring at a loading spinner.

### Why Streaming Matters:
- **Instant Time-To-First-Token (TTFT)**: The user sees text immediately (usually within 1 second).
- **Perceived Performance**: The output appears chunk-by-chunk in real time as Claude thinks and generates.
- **Single HTTP Connection**: The server maintains an open Server-Sent Events (SSE) connection over which events arrive incrementally.

In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Initialize Anthropic Client
from anthropic import Anthropic

client = Anthropic()
model = "claude-3-5-sonnet-latest"

In [ ]:
# Helper functions for message management
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

--- 
## Method 1: Low-Level Event Streaming (`stream=True`)

Setting `stream=True` on `client.messages.create()` returns an iterator yielding raw SSE events.

### Common Event Sequence:
1. `RawMessageStartEvent`: Message initiated with metadata.
2. `RawContentBlockStartEvent`: New content block begun (text, tool call, etc.).
3. `RawContentBlockDeltaEvent`: **Contains the actual text chunk** being generated.
4. `RawContentBlockStopEvent`: Current content block completed.
5. `RawMessageDeltaEvent`: Message metadata update (stop_reason, usage).
6. `RawMessageStopEvent`: End of information for the current message.

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

--- 
## Method 2: High-Level Stream Context Manager (`client.messages.stream`)

Instead of manually parsing event objects, the Anthropic SDK provides a streamlined context manager: `client.messages.stream()`.

By iterating over `stream.text_stream`, you receive **only the generated text deltas**, making it trivial to render in real-time.

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
print()

--- 
## Method 3: Getting the Final Assembled Message (`get_final_message()`)

While streaming is essential for the front-end user experience, back-end servers frequently need the **complete, final message** (to persist to a database or append to multi-turn conversation history).

Calling `stream.get_final_message()` automatically gathers all deltas into a standard, fully populated `Message` object.

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        # Stream to client UI in production
        print(text, end="", flush=True)

print("\n\n--- Gathering Final Message ---")
final_message = stream.get_final_message()
print("Stop Reason:", final_message.stop_reason)
print("Input Tokens:", final_message.usage.input_tokens)
print("Output Tokens:", final_message.usage.output_tokens)
print("Full Content:", final_message.content[0].text)